In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip /content/drive/MyDrive/Dataset.zip -d /content/dataset

Streaming output truncated to the last 5000 lines.
  inflating: /content/dataset/Dataset/HAM10000_images_part_2/ISIC_0029397.jpg  
  inflating: /content/dataset/Dataset/HAM10000_images_part_2/ISIC_0029398.jpg  
  inflating: /content/dataset/Dataset/HAM10000_images_part_2/ISIC_0029413.jpg  
  inflating: /content/dataset/Dataset/HAM10000_images_part_2/ISIC_0029427.jpg  
  inflating: /content/dataset/Dataset/HAM10000_images_part_2/ISIC_0029437.jpg  
  inflating: /content/dataset/Dataset/HAM10000_images_part_2/ISIC_0029443.jpg  
  inflating: /content/dataset/Dataset/HAM10000_images_part_2/ISIC_0029447.jpg  
  inflating: /content/dataset/Dataset/HAM10000_images_part_2/ISIC_0029451.jpg  
  inflating: /content/dataset/Dataset/HAM10000_images_part_2/ISIC_0029454.jpg  
  inflating: /content/dataset/Dataset/HAM10000_images_part_2/ISIC_0029457.jpg  
  inflating: /content/dataset/Dataset/HAM10000_images_part_2/ISIC_0029460.jpg  
  inflating: /content/dataset/Dataset/HAM10000_images_part_2/ISIC_002

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.optimizers import Adam

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import BatchNormalization

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications import DenseNet121

In [2]:
data = pd.read_csv("dataset/Dataset/HAM10000_metadata.csv")

In [ ]:
data

,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear
...,...,...,...,...,...,...,...
10010,HAM_0002867,ISIC_0033084,akiec,histo,40.0,male,abdomen
10011,HAM_0002867,ISIC_0033550,akiec,histo,40.0,male,abdomen
10012,HAM_0002867,ISIC_0033536,akiec,histo,40.0,male,abdomen
10013,HAM_0000239,ISIC_0032854,akiec,histo,80.0,male,face


In [3]:
import os
image_dir1 = "dataset/Dataset/HAM10000_images_part_1"
image_dir2 = "dataset/Dataset/HAM10000_images_part_2"

image_paths = {}

for folder in [image_dir1, image_dir2]:

    for img in os.listdir(folder):

        image_id = img.split(".")[0]

        image_paths[image_id] = os.path.join(folder,img)

data["path"] = data["image_id"].map(image_paths)

data.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization,path
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,dataset/Dataset/HAM10000_images_part_1/ISIC_00...
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,dataset/Dataset/HAM10000_images_part_1/ISIC_00...
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,dataset/Dataset/HAM10000_images_part_1/ISIC_00...
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,dataset/Dataset/HAM10000_images_part_1/ISIC_00...
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,dataset/Dataset/HAM10000_images_part_2/ISIC_00...


In [ ]:
train_df, temp_df = train_test_split(data, test_size=0.3, random_state=42, stratify=data["dx"] )

In [ ]:
valid_df, test_df = train_test_split(temp_df , test_size= 0.5, random_state= 42, stratify=temp_df["dx"])

In [ ]:
train_df

,lesion_id,image_id,dx,dx_type,age,sex,localization,path
4357,HAM_0000946,ISIC_0031775,nv,follow_up,60.0,male,trunk,dataset/Dataset/HAM10000_images_part_2/ISIC_00...
1751,HAM_0006097,ISIC_0027306,mel,histo,60.0,male,chest,dataset/Dataset/HAM10000_images_part_1/ISIC_00...
9527,HAM_0004348,ISIC_0033895,nv,consensus,40.0,female,unknown,dataset/Dataset/HAM10000_images_part_2/ISIC_00...
8311,HAM_0006608,ISIC_0025491,nv,histo,60.0,male,back,dataset/Dataset/HAM10000_images_part_1/ISIC_00...
1214,HAM_0005678,ISIC_0031023,mel,histo,60.0,male,chest,dataset/Dataset/HAM10000_images_part_2/ISIC_00...
...,...,...,...,...,...,...,...,...
492,HAM_0001605,ISIC_0024422,bkl,histo,75.0,male,upper extremity,dataset/Dataset/HAM10000_images_part_1/ISIC_00...
7092,HAM_0005143,ISIC_0030303,nv,histo,40.0,male,chest,dataset/Dataset/HAM10000_images_part_2/ISIC_00...
9254,HAM_0002364,ISIC_0029838,nv,consensus,5.0,female,hand,dataset/Dataset/HAM10000_images_part_2/ISIC_00...
5674,HAM_0005583,ISIC_0025574,nv,follow_up,50.0,male,abdomen,dataset/Dataset/HAM10000_images_part_1/ISIC_00...


In [ ]:
valid_df

,lesion_id,image_id,dx,dx_type,age,sex,localization,path
1800,HAM_0002841,ISIC_0032982,mel,histo,20.0,male,back,dataset/Dataset/HAM10000_images_part_2/ISIC_00...
4180,HAM_0000885,ISIC_0025293,nv,follow_up,40.0,female,upper extremity,dataset/Dataset/HAM10000_images_part_1/ISIC_00...
3249,HAM_0004544,ISIC_0031099,nv,follow_up,50.0,male,foot,dataset/Dataset/HAM10000_images_part_2/ISIC_00...
624,HAM_0002548,ISIC_0028656,bkl,histo,50.0,female,lower extremity,dataset/Dataset/HAM10000_images_part_1/ISIC_00...
319,HAM_0007260,ISIC_0028336,bkl,histo,60.0,male,chest,dataset/Dataset/HAM10000_images_part_1/ISIC_00...
...,...,...,...,...,...,...,...,...
6954,HAM_0000324,ISIC_0029945,nv,histo,80.0,male,back,dataset/Dataset/HAM10000_images_part_2/ISIC_00...
1473,HAM_0005490,ISIC_0033931,mel,histo,70.0,male,upper extremity,dataset/Dataset/HAM10000_images_part_2/ISIC_00...
4717,HAM_0001514,ISIC_0024659,nv,follow_up,60.0,male,trunk,dataset/Dataset/HAM10000_images_part_1/ISIC_00...
4366,HAM_0002654,ISIC_0025698,nv,follow_up,40.0,female,trunk,dataset/Dataset/HAM10000_images_part_1/ISIC_00...


# **Encode**

In [4]:
label_encoder = LabelEncoder()

train_df["label"] = label_encoder.fit_transform(train_df["dx"])
valid_df["label"] = label_encoder.transform(valid_df["dx"])
test_df["label"] = label_encoder.transform(test_df["dx"])

NUM_CLASSES = len(label_encoder.classes_)

NameError: name 'train_df' is not defined

In [ ]:
NUM_CLASSES

7

## Compute Weight

In [ ]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)

class_weights = dict(enumerate(class_weights))

print(class_weights)

{0: np.float64(4.37305053025577), 1: np.float64(2.7817460317460316), 2: np.float64(1.3022478172023035), 3: np.float64(12.36331569664903), 4: np.float64(1.285530900421786), 5: np.float64(0.21338772031292808), 6: np.float64(10.115440115440116)}


# Create Image Loader

In [ ]:
IMG_SIZE = (224, 224)

def process_image(path, label):

    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)

    image = tf.image.resize(image, IMG_SIZE)

    return image, label

## **Augumentation**

In [ ]:
data_augmentation = tf.keras.Sequential([

    tf.keras.layers.RandomFlip("horizontal"),

    tf.keras.layers.RandomRotation(0.2),

    tf.keras.layers.RandomZoom(0.2),

    tf.keras.layers.RandomContrast(0.2)

])

In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices(
    (train_df["path"], train_df["label"])
)

train_dataset = train_dataset.map(
    process_image,
    num_parallel_calls=tf.data.AUTOTUNE
)
train_dataset = train_dataset.batch(32)
train_dataset = train_dataset.map(
    lambda x, y: (data_augmentation(x, training=True), y),
    num_parallel_calls=tf.data.AUTOTUNE
)
train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)

In [ ]:
valid_dataset = tf.data.Dataset.from_tensor_slices(
    (valid_df["path"], valid_df["label"])
)

valid_dataset = valid_dataset.map(
    process_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

BATCH_SIZE = 32

valid_dataset = valid_dataset.batch(BATCH_SIZE)

valid_dataset = valid_dataset.prefetch(tf.data.AUTOTUNE)

In [ ]:
test_dataset = tf.data.Dataset.from_tensor_slices(
    (test_df["path"], test_df["label"])
)

test_dataset = test_dataset.map(
    process_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

BATCH_SIZE = 32

test_dataset = test_dataset.batch(BATCH_SIZE)

test_dataset = test_dataset.prefetch(tf.data.AUTOTUNE)

In [ ]:
basic_cnn = tf.keras.Sequential([

    tf.keras.layers.Input(shape=(224,224,3)),

    tf.keras.layers.Conv2D(
        32,
        (3,3),
        activation="relu"
    ),

    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Conv2D(
        64,
        (3,3),
        activation="relu"
    ),

    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Conv2D(
            128,
            (3,3),
            activation="relu"
        ),

    tf.keras.layers.MaxPooling2D((2,2)),


    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(
        256,
        activation="relu"
    ),

    tf.keras.layers.Dense(
        NUM_CLASSES,
        activation="softmax"
    )

])

In [ ]:
basic_cnn.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │    22,151,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,246,471 (84.86 MB)

 Trainable params: 22,246,471 (84.86 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
basic_cnn.compile(

    optimizer="adam",

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)

history_basic = basic_cnn.fit(

    train_dataset,

    validation_data=valid_dataset,

    epochs=5,

    class_weight=class_weights,

)

Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 126s 527ms/step - accuracy: 0.1576 - loss: 19.3815 - val_accuracy: 0.0632 - val_loss: 1.9803
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 126s 570ms/step - accuracy: 0.0611 - loss: 2.4126 - val_accuracy: 0.0686 - val_loss: 1.9856
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 108s 485ms/step - accuracy: 0.0608 - loss: 1.9557 - val_accuracy: 0.0553 - val_loss: 1.9741
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 109s 493ms/step - accuracy: 0.0492 - loss: 1.9394 - val_accuracy: 0.0573 - val_loss: 1.9618
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 105s 478ms/step - accuracy: 0.0586 - loss: 2.8367 - val_accuracy: 0.0573 - val_loss: 1.9650


## **Retrain Because of Low Accuracy**

In [ ]:
basic_cnn = tf.keras.Sequential([

    tf.keras.layers.Input(shape=(224,224,3)),

    tf.keras.layers.Conv2D(
        32,
        (3,3),
        activation="relu"
    ),

    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Conv2D(
        64,
        (3,3),
        activation="relu"
    ),

    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(
        128,
        activation="relu"
    ),

    tf.keras.layers.Dense(
        NUM_CLASSES,
        activation="softmax"
    )

])

basic_cnn.compile(

    optimizer="adam",

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)

history_basic = basic_cnn.fit(

    train_dataset,

    validation_data=valid_dataset,

    epochs=10,

    class_weight=class_weights,

)

Epoch 1/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 128s 550ms/step - accuracy: 0.1151 - loss: 89.3582 - val_accuracy: 0.0706 - val_loss: 1.9517
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 136s 546ms/step - accuracy: 0.0805 - loss: 1.9553 - val_accuracy: 0.0672 - val_loss: 1.9598
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 126s 570ms/step - accuracy: 0.0549 - loss: 1.9317 - val_accuracy: 0.0226 - val_loss: 1.9552
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 115s 521ms/step - accuracy: 0.0437 - loss: 1.9185 - val_accuracy: 0.0333 - val_loss: 1.9477
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 120s 543ms/step - accuracy: 0.0494 - loss: 1.9271 - val_accuracy: 0.0306 - val_loss: 1.9527
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 105s 478ms/step - accuracy: 0.0482 - loss: 2.0073 - val_accuracy: 0.0186 - val_loss: 1.9561
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 141s 472ms/step - accuracy: 0.0342 - loss: 1.9154 - val_accuracy: 0.0266 - val_loss: 1.9434
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 104s 471ms/step - accuracy: 0.0344 

In [ ]:
train_df, temp_df = train_test_split(data, test_size=0.3, random_state=42, stratify=data["dx"] )
valid_df, test_df = train_test_split(temp_df , test_size= 0.5, random_state= 42, stratify=temp_df["dx"])
# ReEncoder

label_encoder = LabelEncoder()

train_df["label"] = label_encoder.fit_transform(train_df["dx"])
valid_df["label"] = label_encoder.transform(valid_df["dx"])
test_df["label"] = label_encoder.transform(test_df["dx"])

NUM_CLASSES = len(label_encoder.classes_)

# Weight

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)

class_weights = dict(enumerate(class_weights))

# Image Loader

IMG_SIZE = (224, 224)

def process_image(path, label):

    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)

    image = tf.cast(image, tf.float32) / 255.0

    return image, label

# Reaugumentation

data_augmentation = tf.keras.Sequential([

    tf.keras.layers.RandomFlip("horizontal"),

    tf.keras.layers.RandomRotation(0.05)

])

# Train
train_dataset = tf.data.Dataset.from_tensor_slices(
    (train_df["path"], train_df["label"])
)

train_dataset = train_dataset.map(
    process_image,
    num_parallel_calls=tf.data.AUTOTUNE
)
train_dataset = train_dataset.shuffle(
    buffer_size=len(train_df),
    reshuffle_each_iteration=True
)

train_dataset = train_dataset.batch(32)

train_dataset = train_dataset.map(
    lambda x, y: (data_augmentation(x, training=True), y),
    num_parallel_calls=tf.data.AUTOTUNE
)
train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)

# Validation

valid_dataset = tf.data.Dataset.from_tensor_slices(
    (valid_df["path"], valid_df["label"])
)

valid_dataset = valid_dataset.map(
    process_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

BATCH_SIZE = 32
valid_dataset = valid_dataset.batch(BATCH_SIZE)
valid_dataset = valid_dataset.cache()
valid_dataset = valid_dataset.prefetch(tf.data.AUTOTUNE)


basic_cnn = tf.keras.Sequential([

    tf.keras.layers.Input(shape=(224,224,3)),

    tf.keras.layers.Conv2D(
        32,
        (3,3),
        activation="relu"
    ),

    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Conv2D(
        64,
        (3,3),
        activation="relu"
    ),

    tf.keras.layers.MaxPooling2D((2,2)),


    tf.keras.layers.Conv2D(
        128,
        (3,3),
        activation="relu"
    ),

    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(
        256,
        activation="relu"
    ),

    tf.keras.layers.Dense(
        NUM_CLASSES,
        activation="softmax"
    )

])

basic_cnn.compile(

    optimizer="adam",

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)

history_basic = basic_cnn.fit(

    train_dataset,

    validation_data=valid_dataset,

    epochs=10,

    class_weight=class_weights,

)

Epoch 1/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 87s 220ms/step - accuracy: 0.3240 - loss: 1.9442 - val_accuracy: 0.4834 - val_loss: 1.3651
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 129s 182ms/step - accuracy: 0.2007 - loss: 1.9021 - val_accuracy: 0.2370 - val_loss: 1.8600
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 79s 173ms/step - accuracy: 0.2545 - loss: 1.7413 - val_accuracy: 0.1152 - val_loss: 3.5857
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 77s 174ms/step - accuracy: 0.3476 - loss: 1.5973 - val_accuracy: 0.5186 - val_loss: 1.3181
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 81s 178ms/step - accuracy: 0.3973 - loss: 1.6254 - val_accuracy: 0.3908 - val_loss: 1.6720
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 84s 182ms/step - accuracy: 0.4478 - loss: 1.4973 - val_accuracy: 0.5146 - val_loss: 1.2791
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 78s 182ms/step - accuracy: 0.4739 - loss: 1.3676 - val_accuracy: 0.4774 - val_loss: 1.3702
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 77s 173ms/step - accuracy: 0.4990 - loss: 

In [ ]:
train_df, temp_df = train_test_split(data, test_size=0.3, random_state=42, stratify=data["dx"] )
valid_df, test_df = train_test_split(temp_df , test_size= 0.5, random_state= 42, stratify=temp_df["dx"])
# ReEncoder

label_encoder = LabelEncoder()

train_df["label"] = label_encoder.fit_transform(train_df["dx"])
valid_df["label"] = label_encoder.transform(valid_df["dx"])
test_df["label"] = label_encoder.transform(test_df["dx"])

NUM_CLASSES = len(label_encoder.classes_)

# Weight

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)

class_weights = dict(enumerate(class_weights))

# Image Loader

IMG_SIZE = (224, 224)

def process_image(path, label):

    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)

    image = tf.cast(image, tf.float32) / 255.0

    return image, label

# Reaugumentation

data_augmentation = tf.keras.Sequential([

    tf.keras.layers.RandomFlip("horizontal"),

    tf.keras.layers.RandomRotation(0.05)

])

# Train
train_dataset = tf.data.Dataset.from_tensor_slices(
    (train_df["path"], train_df["label"])
)

train_dataset = train_dataset.map(
    process_image,
    num_parallel_calls=tf.data.AUTOTUNE
)
train_dataset = train_dataset.shuffle(
    buffer_size=len(train_df),
    reshuffle_each_iteration=True
)

train_dataset = train_dataset.batch(32)

train_dataset = train_dataset.map(
    lambda x, y: (data_augmentation(x, training=True), y),
    num_parallel_calls=tf.data.AUTOTUNE
)
train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)

# Validation

valid_dataset = tf.data.Dataset.from_tensor_slices(
    (valid_df["path"], valid_df["label"])
)

valid_dataset = valid_dataset.map(
    process_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

BATCH_SIZE = 32
valid_dataset = valid_dataset.batch(BATCH_SIZE)
valid_dataset = valid_dataset.cache()
valid_dataset = valid_dataset.prefetch(tf.data.AUTOTUNE)


basic_cnn = tf.keras.Sequential([

    tf.keras.layers.Input(shape=(224,224,3)),

    tf.keras.layers.Conv2D(
        32,
        (3,3),
        activation="relu"
    ),

    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Conv2D(
        64,
        (3,3),
        activation="relu"
    ),

    tf.keras.layers.MaxPooling2D((2,2)),


    tf.keras.layers.Conv2D(
        128,
        (3,3),
        activation="relu"
    ),

    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(
        256,
        activation="relu"
    ),

    tf.keras.layers.Dense(
        NUM_CLASSES,
        activation="softmax"
    )

])

basic_cnn.compile(

    optimizer="adam",

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)

history_basic = basic_cnn.fit(

    train_dataset,

    validation_data=valid_dataset,

    epochs=10,

    class_weight=class_weights,

)

Epoch 1/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 111s 258ms/step - accuracy: 0.3348 - loss: 1.9756 - val_accuracy: 0.0846 - val_loss: 2.4464
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 85s 206ms/step - accuracy: 0.1698 - loss: 1.8826 - val_accuracy: 0.0293 - val_loss: 2.0346
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 77s 198ms/step - accuracy: 0.1260 - loss: 1.8477 - val_accuracy: 0.0220 - val_loss: 2.0675
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 79s 204ms/step - accuracy: 0.1719 - loss: 1.7897 - val_accuracy: 0.0226 - val_loss: 2.0269
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 87s 199ms/step - accuracy: 0.0278 - loss: 1.9407 - val_accuracy: 0.0213 - val_loss: 1.9602
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 94s 212ms/step - accuracy: 0.0475 - loss: 1.9077 - val_accuracy: 0.0686 - val_loss: 1.8653
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 128s 206ms/step - accuracy: 0.1147 - loss: 1.9284 - val_accuracy: 0.0360 - val_loss: 1.9543
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 82s 212ms/step - accuracy: 0.1003 - loss:

## **Best Model Basic CNN**

In [ ]:
train_df, temp_df = train_test_split(data, test_size=0.3, random_state=42, stratify=data["dx"] )
valid_df, test_df = train_test_split(temp_df , test_size= 0.5, random_state= 42, stratify=temp_df["dx"])
# ReEncoder

label_encoder = LabelEncoder()

train_df["label"] = label_encoder.fit_transform(train_df["dx"])
valid_df["label"] = label_encoder.transform(valid_df["dx"])
test_df["label"] = label_encoder.transform(test_df["dx"])

NUM_CLASSES = len(label_encoder.classes_)

# Weight

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)

class_weights = dict(enumerate(class_weights))

# Image Loader

IMG_SIZE = (224, 224)

def process_image(path, label):

    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)

    image = tf.cast(image, tf.float32) / 255.0

    return image, label

# Reaugumentation

data_augmentation = tf.keras.Sequential([

    tf.keras.layers.RandomFlip("horizontal"),

    tf.keras.layers.RandomRotation(0.05)

])

# Train
train_dataset = tf.data.Dataset.from_tensor_slices(
    (train_df["path"], train_df["label"])
)

train_dataset = train_dataset.map(
    process_image,
    num_parallel_calls=tf.data.AUTOTUNE
)
train_dataset = train_dataset.shuffle(
    buffer_size = 1000,
    reshuffle_each_iteration=True
)

train_dataset = train_dataset.batch(32)

train_dataset = train_dataset.map(
    lambda x, y: (data_augmentation(x, training=True), y),
    num_parallel_calls=tf.data.AUTOTUNE
)
train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)

# Validation

valid_dataset = tf.data.Dataset.from_tensor_slices(
    (valid_df["path"], valid_df["label"])
)

valid_dataset = valid_dataset.map(
    process_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

BATCH_SIZE = 32
valid_dataset = valid_dataset.batch(BATCH_SIZE)
valid_dataset = valid_dataset.cache()
valid_dataset = valid_dataset.prefetch(tf.data.AUTOTUNE)


basic_cnn = tf.keras.Sequential([

    tf.keras.layers.Input(shape=(224,224,3)),

    tf.keras.layers.Conv2D(
        32,
        (3,3),
        activation="relu"
    ),

    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Conv2D(
        64,
        (3,3),
        activation="relu"
    ),

    tf.keras.layers.MaxPooling2D((2,2)),


    tf.keras.layers.Conv2D(
        128,
        (3,3),
        activation="relu"
    ),

    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(
        256,
        activation="relu"
    ),

    tf.keras.layers.Dense(
        NUM_CLASSES,
        activation="softmax"
    )

])

basic_cnn.compile(

    optimizer="adam",

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)

history_basic = basic_cnn.fit(

    train_dataset,

    validation_data=valid_dataset,

    epochs=20,

    class_weight=class_weights,

)

Epoch 1/20
220/220 ━━━━━━━━━━━━━━━━━━━━ 91s 358ms/step - accuracy: 0.3767 - loss: 1.8434 - val_accuracy: 0.4640 - val_loss: 1.3469
Epoch 2/20
220/220 ━━━━━━━━━━━━━━━━━━━━ 76s 313ms/step - accuracy: 0.4518 - loss: 1.5414 - val_accuracy: 0.0992 - val_loss: 2.3007
Epoch 3/20
220/220 ━━━━━━━━━━━━━━━━━━━━ 76s 321ms/step - accuracy: 0.4412 - loss: 1.4788 - val_accuracy: 0.4694 - val_loss: 1.4333
Epoch 4/20
220/220 ━━━━━━━━━━━━━━━━━━━━ 77s 328ms/step - accuracy: 0.5118 - loss: 1.3263 - val_accuracy: 0.4574 - val_loss: 1.4000
Epoch 5/20
220/220 ━━━━━━━━━━━━━━━━━━━━ 82s 318ms/step - accuracy: 0.4806 - loss: 1.3315 - val_accuracy: 0.5033 - val_loss: 1.1840
Epoch 6/20
220/220 ━━━━━━━━━━━━━━━━━━━━ 77s 318ms/step - accuracy: 0.4994 - loss: 1.2842 - val_accuracy: 0.2577 - val_loss: 1.6648
Epoch 7/20
220/220 ━━━━━━━━━━━━━━━━━━━━ 76s 315ms/step - accuracy: 0.4892 - loss: 1.2878 - val_accuracy: 0.4068 - val_loss: 1.6359
Epoch 8/20
220/220 ━━━━━━━━━━━━━━━━━━━━ 81s 324ms/step - accuracy: 0.5168 - loss: 1

In [ ]:
import pickle

basic_cnn.save("/content/drive/MyDrive/SkinProject/Basic_cnn/basic_cnn.keras")

with open("/content/drive/MyDrive/SkinProject/Basic_cnn/basic_history.pkl", "wb") as f:
    pickle.dump(history_basic.history, f)

In [ ]:
train_loss,train_acc=basic_cnn.evaluate(train_dataset)
val_loss,val_acc=basic_cnn.evaluate(valid_dataset)
test_loss,test_acc=basic_cnn.evaluate(test_dataset)

220/220 ━━━━━━━━━━━━━━━━━━━━ 74s 316ms/step - accuracy: 0.6385 - loss: 0.8965
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5812 - loss: 1.1924
47/47 ━━━━━━━━━━━━━━━━━━━━ 10s 219ms/step - accuracy: 0.3187 - loss: 243.0634


## Deep Cnn

In [ ]:


deep_cnn = tf.keras.Sequential([

    tf.keras.layers.Input(
        shape=(224,224,3)
    ),

    tf.keras.layers.Conv2D(
        32,
        (3,3),
        activation="relu"
    ),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(
        64,
        (3,3),
        activation="relu"
    ),

    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(
        128,
        (3,3),
        activation="relu"
    ),

    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(
        256,
        activation="relu"
    ),

    tf.keras.layers.Dense(
        NUM_CLASSES,
        activation="softmax"
    )

])

deep_cnn.compile(

    optimizer="adam",

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)

history_deep = deep_cnn.fit(

    train_dataset,

    validation_data=valid_dataset,

    epochs=10,

    class_weight=class_weights,

)

Epoch 1/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 129s 425ms/step - accuracy: 0.0465 - loss: 1.9802 - val_accuracy: 0.1099 - val_loss: 1.9501
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 123s 409ms/step - accuracy: 0.0505 - loss: 1.9463 - val_accuracy: 0.1099 - val_loss: 1.9502
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 79s 340ms/step - accuracy: 0.0863 - loss: 1.9463 - val_accuracy: 0.1099 - val_loss: 1.9476
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 80s 344ms/step - accuracy: 0.1442 - loss: 1.9464 - val_accuracy: 0.1099 - val_loss: 1.9486
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 79s 328ms/step - accuracy: 0.1960 - loss: 1.9463 - val_accuracy: 0.1099 - val_loss: 1.9454
Epoch 6/10
 26/220 ━━━━━━━━━━━━━━━━━━━━ 1:12 372ms/step - accuracy: 0.1029 - loss: 1.6844

KeyboardInterrupt: 

In [ ]:


deep_cnn = tf.keras.Sequential([

    tf.keras.layers.Input(
        shape=(224,224,3)
    ),


    tf.keras.layers.Conv2D(
        32,
        (3,3),
        activation="relu"
    ),

    tf.keras.layers.MaxPooling2D(),


    tf.keras.layers.Conv2D(
        64,
        (3,3),
        activation="relu"
    ),

    tf.keras.layers.MaxPooling2D(),


    tf.keras.layers.Conv2D(
        128,
        (3,3),
        activation="relu"
    ),

    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(
        256,
        (3,3),
        activation="relu"
    ),

    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Flatten(),


    tf.keras.layers.Dense(
        512,
        activation="relu"
    ),


    tf.keras.layers.Dense(
        NUM_CLASSES,
        activation="softmax"
    )

])

deep_cnn.compile(

    optimizer="adam",

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)

history_deep = deep_cnn.fit(

    train_dataset,

    validation_data=valid_dataset,

    epochs=20,

    class_weight=class_weights,

    verbose=1

)

Epoch 1/20
220/220 ━━━━━━━━━━━━━━━━━━━━ 88s 337ms/step - accuracy: 0.3307 - loss: 1.9254 - val_accuracy: 0.1378 - val_loss: 1.9512
Epoch 2/20
220/220 ━━━━━━━━━━━━━━━━━━━━ 79s 337ms/step - accuracy: 0.3153 - loss: 1.8626 - val_accuracy: 0.2537 - val_loss: 1.7969
Epoch 3/20
220/220 ━━━━━━━━━━━━━━━━━━━━ 78s 309ms/step - accuracy: 0.2974 - loss: 1.8187 - val_accuracy: 0.3083 - val_loss: 1.6131
Epoch 4/20
220/220 ━━━━━━━━━━━━━━━━━━━━ 82s 318ms/step - accuracy: 0.3441 - loss: 1.7266 - val_accuracy: 0.3775 - val_loss: 1.4738
Epoch 5/20
220/220 ━━━━━━━━━━━━━━━━━━━━ 84s 323ms/step - accuracy: 0.3943 - loss: 1.6520 - val_accuracy: 0.1937 - val_loss: 1.9817
Epoch 6/20
220/220 ━━━━━━━━━━━━━━━━━━━━ 76s 326ms/step - accuracy: 0.4140 - loss: 1.6390 - val_accuracy: 0.3609 - val_loss: 1.6024
Epoch 7/20
220/220 ━━━━━━━━━━━━━━━━━━━━ 83s 322ms/step - accuracy: 0.4151 - loss: 1.5972 - val_accuracy: 0.4647 - val_loss: 1.3097
Epoch 8/20
220/220 ━━━━━━━━━━━━━━━━━━━━ 76s 318ms/step - accuracy: 0.4358 - loss: 1

In [ ]:
train_loss,train_acc=deep_cnn.evaluate(train_dataset)
val_loss,val_acc=deep_cnn.evaluate(valid_dataset)
test_loss,test_acc=deep_cnn.evaluate(test_dataset)

220/220 ━━━━━━━━━━━━━━━━━━━━ 73s 310ms/step - accuracy: 0.6026 - loss: 1.0520
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.5473 - loss: 1.2833
47/47 ━━━━━━━━━━━━━━━━━━━━ 8s 173ms/step - accuracy: 0.2342 - loss: 194.9326


In [ ]:
import pickle

deep_cnn.save("/content/drive/MyDrive/SkinProject/deep_cnn/deep_cnn.keras")

with open("/content/drive/MyDrive/SkinProject/deep_cnn/history_deep.pkl", "wb") as f:
    pickle.dump(history_deep.history, f)

## BatchNormalization

In [ ]:

cnn_bn =tf.keras.Sequential([


    tf.keras.layers.Input(
        shape=(224,224,3)
    ),


    tf.keras.layers.Conv2D(
        32,
        (3,3)
    ),

    tf.keras.layers.BatchNormalization(),

    tf.keras.layers.Activation("relu"),

    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(
        64,
        (3,3)
    ),

    tf.keras.layers.BatchNormalization(),

    tf.keras.layers.Activation("relu"),


    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(
        128,
        (3,3)
    ),

    tf.keras.layers.BatchNormalization(),

    tf.keras.layers.Activation("relu"),


    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Flatten(),


    tf.keras.layers.Dense(
        256,
        activation="relu"
    ),


    tf.keras.layers.Dense(
        NUM_CLASSES,
        activation="softmax"
    )

])

cnn_bn.compile(

optimizer="adam",

loss="sparse_categorical_crossentropy",

metrics=["accuracy"]

)


history_bn=cnn_bn.fit(

train_dataset,

validation_data=valid_dataset,

epochs=10,

class_weight=class_weights

)

Epoch 1/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 95s 367ms/step - accuracy: 0.2411 - loss: 8.7828 - val_accuracy: 0.1099 - val_loss: 2.6164
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 128s 321ms/step - accuracy: 0.4131 - loss: 1.6675 - val_accuracy: 0.2976 - val_loss: 2.4919
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 77s 322ms/step - accuracy: 0.5113 - loss: 1.5594 - val_accuracy: 0.5799 - val_loss: 1.7610
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 76s 318ms/step - accuracy: 0.5272 - loss: 1.5374 - val_accuracy: 0.4441 - val_loss: 1.9419
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 80s 315ms/step - accuracy: 0.5561 - loss: 1.4552 - val_accuracy: 0.5952 - val_loss: 1.6296
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 81s 339ms/step - accuracy: 0.5411 - loss: 1.4961 - val_accuracy: 0.4554 - val_loss: 2.4897
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 80s 328ms/step - accuracy: 0.5180 - loss: 1.4749 - val_accuracy: 0.5160 - val_loss: 1.7501
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 75s 322ms/step - accuracy: 0.5475 - loss: 

In [ ]:
train_loss,train_acc=cnn_bn.evaluate(train_dataset)
val_loss,val_acc=cnn_bn.evaluate(valid_dataset)
test_loss,test_acc=cnn_bn.evaluate(test_dataset)

220/220 ━━━━━━━━━━━━━━━━━━━━ 90s 381ms/step - accuracy: 0.5512 - loss: 1.5212
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.5606 - loss: 1.5704
47/47 ━━━━━━━━━━━━━━━━━━━━ 11s 240ms/step - accuracy: 0.6687 - loss: 1.8354


In [ ]:
import pickle

cnn_bn.save("/content/drive/MyDrive/SkinProject/Cnn_bn/cnn_bn.keras")

with open("/content/drive/MyDrive/SkinProject/Cnn_bn/history_bn.pkl", "wb") as f:
    pickle.dump(history_bn.history, f)

## **CNN Dropout**

In [ ]:
cnn_dp=tf.keras.Sequential([


    tf.keras.layers.Input(
        shape=(224,224,3)
    ),


    tf.keras.layers.Conv2D(
        32,
        (3,3),
        activation="relu"
),

    tf.keras.layers.MaxPooling2D(2,2),



    tf.keras.layers.Conv2D(
        64,
        (3,3),
        activation="relu"
    ),

    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(
            128,
            (3,3),
            activation="relu"
        ),

        tf.keras.layers.MaxPooling2D(2,2),


    tf.keras.layers.Flatten(),


    tf.keras.layers.Dense(
        256,
        activation="relu"
    ),


    tf.keras.layers.Dropout(0.5),


    tf.keras.layers.Dense(
        NUM_CLASSES,
        activation="softmax"
    )

])

cnn_dp.compile(

optimizer="adam",

loss="sparse_categorical_crossentropy",

metrics=["accuracy"]

)


history_dp =cnn_dp.fit(

train_dataset,

validation_data=valid_dataset,

epochs=10,

class_weight=class_weights

)

Epoch 1/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 86s 332ms/step - accuracy: 0.2789 - loss: 2.0305 - val_accuracy: 0.1099 - val_loss: 1.9692
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 75s 311ms/step - accuracy: 0.3767 - loss: 1.8461 - val_accuracy: 0.1005 - val_loss: 2.4781
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 75s 315ms/step - accuracy: 0.2541 - loss: 1.8704 - val_accuracy: 0.0426 - val_loss: 1.8922
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 80s 313ms/step - accuracy: 0.0561 - loss: 1.9349 - val_accuracy: 0.0333 - val_loss: 1.9902
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 75s 312ms/step - accuracy: 0.2340 - loss: 1.8633 - val_accuracy: 0.0360 - val_loss: 2.0438
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 75s 313ms/step - accuracy: 0.2531 - loss: 1.7777 - val_accuracy: 0.0486 - val_loss: 2.0160
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 80s 314ms/step - accuracy: 0.3251 - loss: 1.7211 - val_accuracy: 0.2783 - val_loss: 1.7223
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 74s 311ms/step - accuracy: 0.3675 - loss: 1

In [ ]:
train_loss,train_acc=cnn_dp.evaluate(train_dataset)
val_loss,val_acc=cnn_dp.evaluate(valid_dataset)
test_loss,test_acc=cnn_dp.evaluate(test_dataset)

220/220 ━━━━━━━━━━━━━━━━━━━━ 86s 347ms/step - accuracy: 0.4718 - loss: 1.3425
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.4414 - loss: 1.4081
47/47 ━━━━━━━━━━━━━━━━━━━━ 8s 160ms/step - accuracy: 0.1424 - loss: 114.2452


In [ ]:
import pickle

cnn_dp.save("/content/drive/MyDrive/SkinProject/Cnn_dp/cnn_dp.keras")

with open("/content/drive/MyDrive/SkinProject/Cnn_dp/history_dp.pkl", "wb") as f:
    pickle.dump(history_dp.history, f)

#  **Transfer Learning Techniques**

In [6]:
def build_transfer_model(base_model):


    base_model.trainable=False


    model=tf.keras.Sequential([


        base_model,


        tf.keras.layers.GlobalAveragePooling2D(),


        tf.keras.layers.Dense(
            256,
            activation="relu"
        ),


        tf.keras.layers.Dropout(0.5),


        tf.keras.layers.Dense(
            NUM_CLASSES,
            activation="softmax"
        )

    ])


    return model

## **Mobile Net**

In [ ]:
mobilenet_base=tf.keras.applications.MobileNetV2(

    input_shape=(224,224,3),

    include_top=False,

    weights="imagenet"

)

mobilenet_model=build_transfer_model(
    mobilenet_base
)

mobilenet_model.compile(

optimizer="adam",

loss="sparse_categorical_crossentropy",

metrics=["accuracy"]

)


history_mobile= mobilenet_model.fit(

train_dataset,

validation_data=valid_dataset,

epochs=10,

class_weight=class_weights

)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 134s 473ms/step - accuracy: 0.3899 - loss: 1.7463 - val_accuracy: 0.4567 - val_loss: 1.4227
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 78s 321ms/step - accuracy: 0.4856 - loss: 1.4000 - val_accuracy: 0.3988 - val_loss: 1.3972
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 76s 318ms/step - accuracy: 0.5061 - loss: 1.2954 - val_accuracy: 0.3961 - val_loss: 1.4325
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 81s 322ms/step - accuracy: 0.5074 - loss: 1.2298 - val_accuracy: 0.5393 - val_loss: 1.1498
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 74s 313ms/step - accuracy: 0.5352 - loss: 1.1999 - val_accuracy: 0.5726 - val_loss: 1.1300
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 84s 321ms/step - accuracy: 0.5291 - loss: 1.1660 - val_accuracy: 0.6538 - val_loss: 0.9169
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 76s 324ms/step - accuracy: 0.5352 - loss: 1.1399 - val_accuracy: 0.5786 - val_loss: 1.0763
Epoch 8/10
220/220 ━━━━━━━━━━━━━━

In [ ]:
train_loss,train_acc=mobilenet_model.evaluate(train_dataset)
val_loss,val_acc=mobilenet_model.evaluate(valid_dataset)
test_loss,test_acc=mobilenet_model.evaluate(test_dataset)

220/220 ━━━━━━━━━━━━━━━━━━━━ 79s 337ms/step - accuracy: 0.6244 - loss: 0.9713
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.6059 - loss: 1.0230
47/47 ━━━━━━━━━━━━━━━━━━━━ 22s 473ms/step - accuracy: 0.2741 - loss: 1.9082


In [ ]:
import pickle

mobilenet_model.save("/content/drive/MyDrive/SkinProject/Mobilenet_Model/mobilenet_model.keras")

with open("/content/drive/MyDrive/SkinProject/Mobilenet_Model/history_mobile.pkl", "wb") as f:
    pickle.dump(history_mobile.history, f)

In [ ]:
train_df, temp_df = train_test_split(data, test_size=0.3, random_state=42, stratify=data["dx"] )
valid_df, test_df = train_test_split(temp_df , test_size= 0.5, random_state= 42, stratify=temp_df["dx"])
# ReEncoder

label_encoder = LabelEncoder()

train_df["label"] = label_encoder.fit_transform(train_df["dx"])
valid_df["label"] = label_encoder.transform(valid_df["dx"])
test_df["label"] = label_encoder.transform(test_df["dx"])

NUM_CLASSES = len(label_encoder.classes_)

# Weight

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)

class_weights = dict(enumerate(class_weights))

# Image Loader

IMG_SIZE = (224, 224)

def process_image(path, label):

    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)

    image = tf.cast(image, tf.float32) / 255.0

    return image, label

# Reaugumentation

data_augmentation = tf.keras.Sequential([

    tf.keras.layers.RandomFlip("horizontal"),

    tf.keras.layers.RandomRotation(0.05)

])

# Train
train_dataset = tf.data.Dataset.from_tensor_slices(
    (train_df["path"], train_df["label"])
)

train_dataset = train_dataset.map(
    process_image,
    num_parallel_calls=tf.data.AUTOTUNE
)
train_dataset = train_dataset.shuffle(
    buffer_size=len(train_df),
    reshuffle_each_iteration=True
)

train_dataset = train_dataset.batch(32)

train_dataset = train_dataset.map(
    lambda x, y: (data_augmentation(x, training=True), y),
    num_parallel_calls=tf.data.AUTOTUNE
)
train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)

# Validation

valid_dataset = tf.data.Dataset.from_tensor_slices(
    (valid_df["path"], valid_df["label"])
)

valid_dataset = valid_dataset.map(
    process_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

BATCH_SIZE = 32
valid_dataset = valid_dataset.batch(BATCH_SIZE)
valid_dataset = valid_dataset.cache()
valid_dataset = valid_dataset.prefetch(tf.data.AUTOTUNE)

# **Efficient Net**

In [ ]:

efficient_base=tf.keras.applications.EfficientNetB0(

input_shape=(224,224,3),

include_top=False,

weights="imagenet"

)


efficient_model=build_transfer_model(
    efficient_base
)

# compile

efficient_model.compile(

optimizer="adam",

loss="sparse_categorical_crossentropy",

metrics=["accuracy"]

)

# Train

history_efficient= efficient_model.fit(

train_dataset,

validation_data=valid_dataset,

epochs=10,

class_weight=class_weights

)

Epoch 1/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 159s 398ms/step - accuracy: 0.0884 - loss: 2.0084 - val_accuracy: 0.0113 - val_loss: 1.9489
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 79s 191ms/step - accuracy: 0.0264 - loss: 1.9481 - val_accuracy: 0.0326 - val_loss: 1.9493
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 76s 187ms/step - accuracy: 0.0856 - loss: 1.9474 - val_accuracy: 0.0326 - val_loss: 1.9474
Epoch 4/10
211/220 ━━━━━━━━━━━━━━━━━━━━ 1s 185ms/step - accuracy: 0.0316 - loss: 1.9650

KeyboardInterrupt: 

In [ ]:
train_df, temp_df = train_test_split(data, test_size=0.3, random_state=42, stratify=data["dx"] )
valid_df, test_df = train_test_split(temp_df , test_size= 0.5, random_state= 42, stratify=temp_df["dx"])
# ReEncoder

label_encoder = LabelEncoder()

train_df["label"] = label_encoder.fit_transform(train_df["dx"])
valid_df["label"] = label_encoder.transform(valid_df["dx"])
test_df["label"] = label_encoder.transform(test_df["dx"])

NUM_CLASSES = len(label_encoder.classes_)

# Weight

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)

class_weights = dict(enumerate(class_weights))

# Image Process
def process_image(path, label):

    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, (224,224))
    image = tf.cast(image, tf.float32) / 255.0

    return image, label

# Agumentation
data_augmentation = tf.keras.Sequential([

    tf.keras.layers.RandomFlip("horizontal"),

    tf.keras.layers.RandomRotation(0.05),

    tf.keras.layers.RandomZoom(0.1),

    tf.keras.layers.RandomContrast(0.1)

])# Train
train_dataset = tf.data.Dataset.from_tensor_slices(
    (train_df["path"], train_df["label"])
)

train_dataset = train_dataset.map(
    process_image,
    num_parallel_calls=tf.data.AUTOTUNE
)
train_dataset = train_dataset.shuffle(
    buffer_size=len(train_df),
    reshuffle_each_iteration=True
)

train_dataset = train_dataset.batch(32)

train_dataset = train_dataset.map(
    lambda x, y: (data_augmentation(x, training=True), y),
    num_parallel_calls=tf.data.AUTOTUNE
)
train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)

# Validation

valid_dataset = tf.data.Dataset.from_tensor_slices(
    (valid_df["path"], valid_df["label"])
)

valid_dataset = valid_dataset.map(
    process_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

BATCH_SIZE = 32
valid_dataset = valid_dataset.batch(BATCH_SIZE)
valid_dataset = valid_dataset.cache()
valid_dataset = valid_dataset.prefetch(tf.data.AUTOTUNE)



efficient_base = tf.keras.applications.EfficientNetB0(

    include_top=False,
    weights="imagenet",
    input_shape=(224,224,3)
)

efficient_base.trainable = False

inputs = tf.keras.Input(shape=(224,224,3))

x = efficient_base(inputs, training=False)

x = tf.keras.layers.GlobalAveragePooling2D()(x)

x = tf.keras.layers.BatchNormalization()(x)

x = tf.keras.layers.Dropout(0.3)(x)

outputs = tf.keras.layers.Dense(
    NUM_CLASSES,
    activation="softmax"
)(x)

efficient_model = tf.keras.Model(inputs, outputs)


# compile

efficient_model.compile(

    optimizer="adam",

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)

efficient_history = efficient_model.fit(

    train_dataset,

    validation_data=valid_dataset,

    epochs=10,

    class_weight=class_weights,
)

Epoch 1/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 185s 518ms/step - accuracy: 0.1291 - loss: 2.1574 - val_accuracy: 0.6698 - val_loss: 1.8524
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 107s 323ms/step - accuracy: 0.1335 - loss: 2.1389 - val_accuracy: 0.6198 - val_loss: 1.8858
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 139s 329ms/step - accuracy: 0.1097 - loss: 2.1469 - val_accuracy: 0.6698 - val_loss: 1.8545
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 104s 326ms/step - accuracy: 0.1227 - loss: 2.1893 - val_accuracy: 0.0326 - val_loss: 1.9194
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 104s 329ms/step - accuracy: 0.1193 - loss: 2.1526 - val_accuracy: 0.0140 - val_loss: 1.9357
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 107s 318ms/step - accuracy: 0.1277 - loss: 2.1486 - val_accuracy: 0.2463 - val_loss: 1.8428
Epoch 7/10
 82/220 ━━━━━━━━━━━━━━━━━━━━ 48s 351ms/step - accuracy: 0.1297 - loss: 2.0231

KeyboardInterrupt: 

In [ ]:
import tensorflow as tf
import numpy as np

# =====================================================
# Split Dataset
# =====================================================

train_df, temp_df = train_test_split(
    data,
    test_size=0.30,
    random_state=42,
    stratify=data["dx"]
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["dx"]
)

# =====================================================
# Label Encoding
# =====================================================

label_encoder = LabelEncoder()

train_df["label"] = label_encoder.fit_transform(train_df["dx"])
valid_df["label"] = label_encoder.transform(valid_df["dx"])
test_df["label"] = label_encoder.transform(test_df["dx"])

NUM_CLASSES = len(label_encoder.classes_)

# =====================================================
# Class Weights
# =====================================================

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)

class_weights = dict(enumerate(class_weights))

# =====================================================
# Image Processing
# =====================================================

IMG_SIZE = (224, 224)

def process_image(path, label):

    image = tf.io.read_file(path)

    image = tf.image.decode_jpeg(
        image,
        channels=3
    )

    image = tf.image.resize(
        image,
        IMG_SIZE
    )

    image = tf.cast(
        image,
        tf.float32
    ) 

    return image, label

# =====================================================
# Data Augmentation
# =====================================================

data_augmentation = tf.keras.Sequential([

    tf.keras.layers.RandomFlip("horizontal"),

    tf.keras.layers.RandomRotation(0.05),

    tf.keras.layers.RandomZoom(0.10),

    tf.keras.layers.RandomContrast(0.10)

])

# =====================================================
# Training Dataset
# =====================================================

BATCH_SIZE = 32

train_dataset = tf.data.Dataset.from_tensor_slices(
    (
        train_df["path"],
        train_df["label"]
    )
)

train_dataset = train_dataset.map(
    process_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

train_dataset = train_dataset.shuffle(
    len(train_df)
)

train_dataset = train_dataset.batch(BATCH_SIZE)

train_dataset = train_dataset.map(

    lambda x, y: (
        data_augmentation(
            x,
            training=True
        ),
        y
    ),

    num_parallel_calls=tf.data.AUTOTUNE

)

train_dataset = train_dataset.prefetch(
    tf.data.AUTOTUNE
)

# =====================================================
# Validation Dataset
# =====================================================

valid_dataset = tf.data.Dataset.from_tensor_slices(
    (
        valid_df["path"],
        valid_df["label"]
    )
)

valid_dataset = valid_dataset.map(
    process_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

valid_dataset = valid_dataset.batch(
    BATCH_SIZE
)

valid_dataset = valid_dataset.cache()

valid_dataset = valid_dataset.prefetch(
    tf.data.AUTOTUNE
)

# =====================================================
# EfficientNetB0
# =====================================================

efficient_base = tf.keras.applications.EfficientNetB0(

    include_top=False,

    weights="imagenet",

    input_shape=(224,224,3)

)

efficient_base.trainable = False

# =====================================================
# Build Model
# =====================================================

inputs = tf.keras.Input(
    shape=(224,224,3)
)

x = efficient_base(
    inputs,
    training=False
)

x = tf.keras.layers.GlobalAveragePooling2D()(x)

x = tf.keras.layers.BatchNormalization()(x)

x = tf.keras.layers.Dense(
    256,
    activation="relu"
)(x)

x = tf.keras.layers.Dropout(0.3)(x)

outputs = tf.keras.layers.Dense(
    NUM_CLASSES,
    activation="softmax"
)(x)

efficient_model = tf.keras.Model(
    inputs,
    outputs
)

# =====================================================
# Compile
# =====================================================

efficient_model.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.01
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)

efficient_model.summary()

# =====================================================
# Train
# =====================================================

history_efficient = efficient_model.fit(

    train_dataset,

    validation_data=valid_dataset,

    epochs=10,

    class_weight=class_weights,

    verbose=1

)

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,384,426 (16.73 MB)

 Trainable params: 332,295 (1.27 MB)

 Non-trainable params: 4,052,131 (15.46 MB)

Epoch 1/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 187s 522ms/step - accuracy: 0.1655 - loss: 2.0828 - val_accuracy: 0.0113 - val_loss: 1.9535
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 134s 332ms/step - accuracy: 0.1397 - loss: 2.0779 - val_accuracy: 0.0113 - val_loss: 1.9796
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 105s 332ms/step - accuracy: 0.1260 - loss: 2.0480 - val_accuracy: 0.1112 - val_loss: 1.9442
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 104s 334ms/step - accuracy: 0.1098 - loss: 2.0537 - val_accuracy: 0.0326 - val_loss: 1.9316
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 117s 393ms/step - accuracy: 0.1203 - loss: 2.0442 - val_accuracy: 0.1198 - val_loss: 1.9190
Epoch 6/10


KeyboardInterrupt: 

In [ ]:
import tensorflow as tf
import numpy as np

# =====================================================
# Split Dataset
# =====================================================

train_df, temp_df = train_test_split(
    data,
    test_size=0.2,
    random_state=42,
    stratify=data["dx"]
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["dx"]
)

# =====================================================
# Label Encoding
# =====================================================

label_encoder = LabelEncoder()

train_df["label"] = label_encoder.fit_transform(train_df["dx"])
valid_df["label"] = label_encoder.transform(valid_df["dx"])
test_df["label"] = label_encoder.transform(test_df["dx"])

NUM_CLASSES = len(label_encoder.classes_)

# =====================================================
# Class Weights
# =====================================================

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)

class_weights = dict(enumerate(class_weights))

# =====================================================
# Image Processing
# =====================================================

IMG_SIZE = (224, 224)

def process_image(path, label):

    image = tf.io.read_file(path)

    image = tf.image.decode_jpeg(
        image,
        channels=3
    )

    image = tf.image.resize(
        image,
        IMG_SIZE
    )

    image = tf.cast(
        image,
        tf.float32
    ) / 255.0

    return image, label

# =====================================================
# Data Augmentation
# =====================================================

data_augmentation = tf.keras.Sequential([

    tf.keras.layers.RandomFlip("horizontal"),

    tf.keras.layers.RandomRotation(0.05),

])

# =====================================================
# Training Dataset
# =====================================================

BATCH_SIZE = 32

train_dataset = tf.data.Dataset.from_tensor_slices(
    (
        train_df["path"],
        train_df["label"]
    )
)

train_dataset = train_dataset.map(
    process_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

train_dataset = train_dataset.shuffle(
    len(train_df)
)

train_dataset = train_dataset.batch(BATCH_SIZE)

train_dataset = train_dataset.map(

    lambda x, y: (
        data_augmentation(
            x,
            training=True
        ),
        y
    ),

    num_parallel_calls=tf.data.AUTOTUNE

)

train_dataset = train_dataset.prefetch(
    tf.data.AUTOTUNE
)

# =====================================================
# Validation Dataset
# =====================================================

valid_dataset = tf.data.Dataset.from_tensor_slices(
    (
        valid_df["path"],
        valid_df["label"]
    )
)

valid_dataset = valid_dataset.map(
    process_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

valid_dataset = valid_dataset.batch(
    BATCH_SIZE
)

valid_dataset = valid_dataset.cache()

valid_dataset = valid_dataset.prefetch(
    tf.data.AUTOTUNE
)

# =====================================================
# EfficientNetB0
# =====================================================

efficient_base = tf.keras.applications.EfficientNetB0(

    include_top=False,

    weights="imagenet",

    input_shape=(224,224,3)

)

efficient_base.trainable = False

# =====================================================
# Build Model
# =====================================================

inputs = tf.keras.Input(
    shape=(224,224,3)
)

x = efficient_base(
    inputs,
    training=False
)

x = tf.keras.layers.GlobalAveragePooling2D()

x = tf.keras.layers.BatchNormalization()

x = tf.keras.layers.Dense(
    256,
    activation="relu"
)(x)

x = tf.keras.layers.Dropout(0.3)

outputs = tf.keras.layers.Dense(
    NUM_CLASSES,
    activation="softmax"
)(x)

efficient_model = tf.keras.Model(
    inputs,
    outputs
)

# =====================================================
# Compile
# =====================================================

efficient_model.compile(

    optimizer="adam",

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)

efficient_model.summary()

# =====================================================
# Train
# =====================================================

history_efficient = efficient_model.fit(

    train_dataset,

    validation_data=valid_dataset,

    epochs=10,

    class_weight=class_weights,

    verbose=1

)

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,384,426 (16.73 MB)

 Trainable params: 332,295 (1.27 MB)

 Non-trainable params: 4,052,131 (15.46 MB)

Epoch 1/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 165s 410ms/step - accuracy: 0.1404 - loss: 2.8113 - val_accuracy: 0.0672 - val_loss: 1.9472
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 79s 186ms/step - accuracy: 0.1237 - loss: 2.5310 - val_accuracy: 0.0140 - val_loss: 1.9793
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 72s 182ms/step - accuracy: 0.1124 - loss: 2.4222 - val_accuracy: 0.0326 - val_loss: 1.9648
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 74s 186ms/step - accuracy: 0.1272 - loss: 2.1596 - val_accuracy: 0.0140 - val_loss: 1.9562
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 75s 186ms/step - accuracy: 0.1177 - loss: 2.0310 - val_accuracy: 0.0146 - val_loss: 1.9634
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 79s 191ms/step - accuracy: 0.0994 - loss: 2.0139 - val_accuracy: 0.0326 - val_loss: 1.9500
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 72s 185ms/step - accuracy: 0.0725 - loss: 1.9748 - val_accuracy: 0.0140 - val_loss: 1.9502
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 79s 203ms/step - accuracy: 0.0829 - loss: 

In [ ]:
print(data.head())

print(data.columns)

print(train_df["path"].head())

print(train_df["dx"].value_counts())

print(train_df["label"].value_counts())

print(train_df.iloc[0]["path"])

     lesion_id      image_id   dx dx_type   age   sex localization  \
0  HAM_0000118  ISIC_0027419  bkl   histo  80.0  male        scalp   
1  HAM_0000118  ISIC_0025030  bkl   histo  80.0  male        scalp   
2  HAM_0002730  ISIC_0026769  bkl   histo  80.0  male        scalp   
3  HAM_0002730  ISIC_0025661  bkl   histo  80.0  male        scalp   
4  HAM_0001466  ISIC_0031633  bkl   histo  75.0  male          ear   

                                                path  
0  dataset/Dataset/HAM10000_images_part_1/ISIC_00...  
1  dataset/Dataset/HAM10000_images_part_1/ISIC_00...  
2  dataset/Dataset/HAM10000_images_part_1/ISIC_00...  
3  dataset/Dataset/HAM10000_images_part_1/ISIC_00...  
4  dataset/Dataset/HAM10000_images_part_2/ISIC_00...  
Index(['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization',
       'path'],
      dtype='object')
4357    dataset/Dataset/HAM10000_images_part_2/ISIC_00...
1751    dataset/Dataset/HAM10000_images_part_1/ISIC_00...
9527    dataset/

In [ ]:
import tensorflow as tf
import numpy as np

# =====================================================
# Split Dataset
# =====================================================

train_df, temp_df = train_test_split(
    data,
    test_size=0.30,
    random_state=42,
    stratify=data["dx"]
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["dx"]
)

# =====================================================
# Label Encoding
# =====================================================

label_encoder = LabelEncoder()

train_df["label"] = label_encoder.fit_transform(train_df["dx"])
valid_df["label"] = label_encoder.transform(valid_df["dx"])
test_df["label"] = label_encoder.transform(test_df["dx"])

NUM_CLASSES = len(label_encoder.classes_)

# =====================================================
# Class Weights
# =====================================================

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)

class_weights = dict(enumerate(class_weights))

# =====================================================
# Image Processing
# =====================================================

IMG_SIZE = (224, 224)

def process_image(path, label):

    image = tf.io.read_file(path)

    image = tf.image.decode_jpeg(
        image,
        channels=3
    )

    image = tf.image.resize(
        image,
        IMG_SIZE
    )

    image = tf.cast(
        image,
        tf.float32
    ) / 255.0

    return image, label

# =====================================================
# Data Augmentation
# =====================================================

data_augmentation = tf.keras.Sequential([

    tf.keras.layers.RandomFlip("horizontal"),

    tf.keras.layers.RandomRotation(0.05),

])

# =====================================================
# Training Dataset
# =====================================================

BATCH_SIZE = 32

train_dataset = tf.data.Dataset.from_tensor_slices(
    (
        train_df["path"],
        train_df["label"]
    )
)

train_dataset = train_dataset.map(
    process_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

train_dataset = train_dataset.shuffle(
    len(train_df)
)

train_dataset = train_dataset.batch(BATCH_SIZE)

train_dataset = train_dataset.map(

    lambda x, y: (
        data_augmentation(
            x,
            training=True
        ),
        y
    ),

    num_parallel_calls=tf.data.AUTOTUNE

)

train_dataset = train_dataset.prefetch(
    tf.data.AUTOTUNE
)

# =====================================================
# Validation Dataset
# =====================================================

valid_dataset = tf.data.Dataset.from_tensor_slices(
    (
        valid_df["path"],
        valid_df["label"]
    )
)

valid_dataset = valid_dataset.map(
    process_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

valid_dataset = valid_dataset.batch(
    BATCH_SIZE
)

valid_dataset = valid_dataset.cache()

valid_dataset = valid_dataset.prefetch(
    tf.data.AUTOTUNE
)

test_dataset = tf.data.Dataset.from_tensor_slices(
    (test_df["path"], test_df["label"])
)

test_dataset = test_dataset.map(
    process_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

BATCH_SIZE = 32

test_dataset = test_dataset.batch(BATCH_SIZE)

test_dataset = test_dataset.prefetch(tf.data.AUTOTUNE)
# =====================================================
# EfficientNetB0
# =====================================================

efficient_base = tf.keras.applications.EfficientNetB0(

    include_top=False,

    weights="imagenet",

    input_shape=(224,224,3)

)

efficient_base.trainable = False

# =====================================================
# Build Model
# =====================================================

inputs = tf.keras.Input(
    shape=(224,224,3)
)

x = efficient_base(
    inputs,
    training=False
)

x = tf.keras.layers.GlobalAveragePooling2D()


x = tf.keras.layers.Dense(
    256,
    activation="relu"
)
x = tf.keras.layers.Dropout(0.3)(x)

outputs = tf.keras.layers.Dense(
    NUM_CLASSES,
    activation="softmax"
)(x)

efficient_model = tf.keras.Model(
    inputs,
    outputs
)

# =====================================================
# Compile
# =====================================================

efficient_model.compile(

    optimizer="adam",

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)

efficient_model.summary()

# =====================================================
# Train
# =====================================================

history_efficient = efficient_model.fit(

    train_dataset,

    validation_data=valid_dataset,

    epochs=10,

    class_weight=class_weights,

)

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,384,426 (16.73 MB)

 Trainable params: 332,295 (1.27 MB)

 Non-trainable params: 4,052,131 (15.46 MB)

Epoch 1/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 162s 393ms/step - accuracy: 0.1312 - loss: 2.7393 - val_accuracy: 0.0140 - val_loss: 2.0130
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 75s 191ms/step - accuracy: 0.1278 - loss: 2.5905 - val_accuracy: 0.0326 - val_loss: 1.9435
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 76s 193ms/step - accuracy: 0.1278 - loss: 2.3935 - val_accuracy: 0.0113 - val_loss: 1.9665
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 80s 184ms/step - accuracy: 0.1058 - loss: 2.1326 - val_accuracy: 0.0113 - val_loss: 1.9529
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 74s 183ms/step - accuracy: 0.1143 - loss: 2.0890 - val_accuracy: 0.0113 - val_loss: 1.9706
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 81s 192ms/step - accuracy: 0.0853 - loss: 1.9962 - val_accuracy: 0.0326 - val_loss: 1.9662
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 77s 192ms/step - accuracy: 0.0608 - loss: 1.9767 - val_accuracy: 0.0113 - val_loss: 1.9666
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 79s 181ms/step - accuracy: 0.0877 - loss: 

In [ ]:
train_loss,train_acc=efficient_model.evaluate(train_dataset)
val_loss,val_acc=efficient_model.evaluate(valid_dataset)
test_loss,test_acc=efficient_model.evaluate(test_dataset)

220/220 ━━━━━━━━━━━━━━━━━━━━ 89s 218ms/step - accuracy: 0.0141 - loss: 1.9528
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - accuracy: 0.0140 - loss: 1.9528
47/47 ━━━━━━━━━━━━━━━━━━━━ 21s 443ms/step - accuracy: 0.0146 - loss: 1.9528


In [ ]:
import pickle

efficient_model.save("/content/drive/MyDrive/SkinProject/EfficientNet_Model/efficient_model.keras")

with open("/content/drive/MyDrive/SkinProject/EfficientNet_Model/history_efficient.pkl", "wb") as f:
    pickle.dump(history_efficient.history, f)

# Resnet

In [ ]:
resnet_base=tf.keras.applications.ResNet50(

input_shape=(224,224,3),

include_top=False,

weights="imagenet"

)


resnet_model=build_transfer_model(
    resnet_base
)

# compile

resnet_model.compile(

optimizer="adam",

loss="sparse_categorical_crossentropy",

metrics=["accuracy"]

)

# Train

history_resnet= resnet_model.fit(

train_dataset,

validation_data=valid_dataset,

epochs=10,

class_weight=class_weights

)

Epoch 1/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 120s 285ms/step - accuracy: 0.0853 - loss: 2.0605 - val_accuracy: 0.0513 - val_loss: 1.9465
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 83s 215ms/step - accuracy: 0.0633 - loss: 1.9484 - val_accuracy: 0.0120 - val_loss: 1.9437
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 80s 216ms/step - accuracy: 0.1482 - loss: 1.9463 - val_accuracy: 0.6691 - val_loss: 1.9418
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 80s 210ms/step - accuracy: 0.0345 - loss: 1.9463 - val_accuracy: 0.6698 - val_loss: 1.9430
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 81s 218ms/step - accuracy: 0.3101 - loss: 1.9463 - val_accuracy: 0.1099 - val_loss: 1.9438
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 82s 221ms/step - accuracy: 0.0345 - loss: 1.9462 - val_accuracy: 0.0513 - val_loss: 1.9459
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 82s 220ms/step - accuracy: 0.0923 - loss: 1.9462 - val_accuracy: 0.0513 - val_loss: 1.9449
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 81s 221ms/step - accuracy: 0.1555 - loss: 

In [ ]:
train_loss,train_acc=resnet_model.evaluate(train_dataset)
val_loss,val_acc=resnet_model.evaluate(valid_dataset)
test_loss,test_acc=resnet_model.evaluate(test_dataset)

220/220 ━━━━━━━━━━━━━━━━━━━━ 81s 224ms/step - accuracy: 0.0116 - loss: 1.9445
47/47 ━━━━━━━━━━━━━━━━━━━━ 4s 91ms/step - accuracy: 0.0113 - loss: 1.9445
47/47 ━━━━━━━━━━━━━━━━━━━━ 14s 303ms/step - accuracy: 0.0113 - loss: 1.9445


In [ ]:
import pickle

resnet_model.save("/content/drive/MyDrive/SkinProject/Resnet_Model/resnet_model.keras")

with open("/content/drive/MyDrive/SkinProject/Resnet_Model/history_resnet.pkl", "wb") as f:
    pickle.dump(history_resnet.history, f)

# **DenseNet**

In [ ]:
import tensorflow as tf
import numpy as np

# =====================================================
# Split Dataset
# =====================================================

train_df, temp_df = train_test_split(
    data,
    test_size=0.30,
    random_state=42,
    stratify=data["dx"]
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["dx"]
)

# =====================================================
# Label Encoding
# =====================================================

label_encoder = LabelEncoder()

train_df["label"] = label_encoder.fit_transform(train_df["dx"])
valid_df["label"] = label_encoder.transform(valid_df["dx"])
test_df["label"] = label_encoder.transform(test_df["dx"])

NUM_CLASSES = len(label_encoder.classes_)

# =====================================================
# Class Weights
# =====================================================

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)

class_weights = dict(enumerate(class_weights))

# =====================================================
# Image Processing
# =====================================================

IMG_SIZE = (224, 224)

def process_image(path, label):

    image = tf.io.read_file(path)

    image = tf.image.decode_jpeg(
        image,
        channels=3
    )

    image = tf.image.resize(
        image,
        IMG_SIZE
    )

    image = tf.cast(
        image,
        tf.float32
    ) 
    return image, label

# =====================================================
# Data Augmentation
# =====================================================

data_augmentation = tf.keras.Sequential([

    tf.keras.layers.RandomFlip("horizontal"),

    tf.keras.layers.RandomRotation(0.05),

])

# =====================================================
# Training Dataset
# =====================================================

BATCH_SIZE = 32

train_dataset = tf.data.Dataset.from_tensor_slices(
    (
        train_df["path"],
        train_df["label"]
    )
)

train_dataset = train_dataset.map(
    process_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

train_dataset = train_dataset.shuffle(
    len(train_df)
)

train_dataset = train_dataset.batch(BATCH_SIZE)

train_dataset = train_dataset.map(

    lambda x, y: (
        data_augmentation(
            x,
            training=True
        ),
        y
    ),

    num_parallel_calls=tf.data.AUTOTUNE

)

train_dataset = train_dataset.prefetch(
    tf.data.AUTOTUNE
)

# =====================================================
# Validation Dataset
# =====================================================

valid_dataset = tf.data.Dataset.from_tensor_slices(
    (
        valid_df["path"],
        valid_df["label"]
    )
)

valid_dataset = valid_dataset.map(
    process_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

valid_dataset = valid_dataset.batch(
    BATCH_SIZE
)

valid_dataset = valid_dataset.cache()

valid_dataset = valid_dataset.prefetch(
    tf.data.AUTOTUNE
)

test_dataset = tf.data.Dataset.from_tensor_slices(
    (test_df["path"], test_df["label"])
)

test_dataset = test_dataset.map(
    process_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

BATCH_SIZE = 32

test_dataset = test_dataset.batch(BATCH_SIZE)

test_dataset = test_dataset.prefetch(tf.data.AUTOTUNE)


In [ ]:
c

Epoch 1/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 157s 429ms/step - accuracy: 0.4054 - loss: 1.6950 - val_accuracy: 0.5573 - val_loss: 1.2489
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 79s 206ms/step - accuracy: 0.5056 - loss: 1.3962 - val_accuracy: 0.5419 - val_loss: 1.2846
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 77s 205ms/step - accuracy: 0.5334 - loss: 1.2928 - val_accuracy: 0.5000 - val_loss: 1.3364
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 78s 208ms/step - accuracy: 0.5476 - loss: 1.2474 - val_accuracy: 0.5999 - val_loss: 1.0568
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 86s 204ms/step - accuracy: 0.5545 - loss: 1.1922 - val_accuracy: 0.6272 - val_loss: 0.9806
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 86s 210ms/step - accuracy: 0.5718 - loss: 1.1570 - val_accuracy: 0.6105 - val_loss: 1.0114
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 141s 211ms/step - accuracy: 0.5685 - loss: 1.1037 - val_accuracy: 0.5672 - val_loss: 1.0782
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 87s 210ms/step - accuracy: 0.5438 - loss:

In [11]:
train_loss,train_acc=densenet_model.evaluate(train_dataset)
val_loss,val_acc=densenet_model.evaluate(valid_dataset)
test_loss,test_acc=densenet_model.evaluate(test_dataset)

220/220 ━━━━━━━━━━━━━━━━━━━━ 100s 258ms/step - accuracy: 0.6051 - loss: 1.0487
47/47 ━━━━━━━━━━━━━━━━━━━━ 4s 75ms/step - accuracy: 0.6158 - loss: 1.0296
47/47 ━━━━━━━━━━━━━━━━━━━━ 26s 551ms/step - accuracy: 0.5735 - loss: 1.1035


In [12]:
import pickle

densenet_model.save("/content/drive/MyDrive/SkinProject/Densenet_Model/densenet_model.keras")

with open("/content/drive/MyDrive/SkinProject/Densenet_Model/history_densenet.pkl", "wb") as f:
    pickle.dump(history_densenet.history, f)